# Mã DES (Data Encryption Standard)

## 1. Giới thiệu

DES (Data Encryption Standard) là một thuật toán mã hóa đối xứng thuộc loại **mã khối (block cipher)**.  
Thuật toán được phát triển vào những năm 1970 bởi IBM và được chuẩn hóa bởi NIST.

DES từng là tiêu chuẩn mã hóa phổ biến trong nhiều năm, nhưng hiện nay đã bị coi là **không còn an toàn** do độ dài khóa ngắn.


### Đặc điểm chính của DES:
- Block size: 64 bit (8 bytes)
- Key size: 64 bit (trong đó 56 bit dùng thực sự, 8 bit parity)
- Số vòng lặp (round): 16


### Nguyên lý hoạt động:

DES sử dụng cấu trúc **Feistel Network**, nghĩa là:
- Dữ liệu được chia thành 2 nửa: Left (L) và Right (R)
- Mỗi vòng sẽ:
  - Áp dụng hàm f lên R
  - XOR với L
  - Hoán đổi vị trí


### Tổng quan quy trình mã hóa:

1. Hoán vị ban đầu (Initial Permutation - IP)
2. Chia block thành L và R
3. Lặp 16 vòng Feistel:
   - Sinh subkey
   - Áp dụng hàm f
4. Ghép lại và hoán vị cuối (Final Permutation - FP)


## 2. Biểu diễn dữ liệu và các bảng trong DES

Trong DES, mọi thao tác đều thực hiện trên **bit (nhị phân)**.  
Do đó, dữ liệu đầu vào cần được chuyển sang dạng bit trước khi xử lý.


### 2.1 Chuyển đổi dữ liệu

- Mỗi ký tự → ASCII → 8 bit
- Một block DES = 64 bit = 8 ký tự

Ví dụ:

'A' → 65 → 01000001

### 2.2 Initial Permutation (IP)

Đây là bước hoán vị ban đầu của block 64 bit.

IP table:

In [1]:
IP = [
58, 50, 42, 34, 26, 18, 10, 2,
60, 52, 44, 36, 28, 20, 12, 4,
62, 54, 46, 38, 30, 22, 14, 6,
64, 56, 48, 40, 32, 24, 16, 8,
57, 49, 41, 33, 25, 17, 9, 1,
59, 51, 43, 35, 27, 19, 11, 3,
61, 53, 45, 37, 29, 21, 13, 5,
63, 55, 47, 39, 31, 23, 15, 7
]

### 2.3 Final Permutation (FP)

FP là nghịch đảo của IP.

In [2]:
FP = [
40, 8, 48, 16, 56, 24, 64, 32,
39, 7, 47, 15, 55, 23, 63, 31,
38, 6, 46, 14, 54, 22, 62, 30,
37, 5, 45, 13, 53, 21, 61, 29,
36, 4, 44, 12, 52, 20, 60, 28,
35, 3, 43, 11, 51, 19, 59, 27,
34, 2, 42, 10, 50, 18, 58, 26,
33, 1, 41, 9, 49, 17, 57, 25
]

### 2.4 Expansion Table (E)

Mở rộng từ 32 bit → 48 bit để XOR với subkey.

In [3]:
E = [
32, 1, 2, 3, 4, 5,
4, 5, 6, 7, 8, 9,
8, 9, 10, 11, 12, 13,
12, 13, 14, 15, 16, 17,
16, 17, 18, 19, 20, 21,
20, 21, 22, 23, 24, 25,
24, 25, 26, 27, 28, 29,
28, 29, 30, 31, 32, 1
]

### 2.5 Permutation P

Dùng sau bước S-box để xáo trộn lại bit.

In [4]:
P = [
16, 7, 20, 21,
29, 12, 28, 17,
1, 15, 23, 26,
5, 18, 31, 10,
2, 8, 24, 14,
32, 27, 3, 9,
19, 13, 30, 6,
22, 11, 4, 25
]

## 3. Key Schedule (Sinh 16 khóa con)

Trong DES, từ khóa ban đầu 64 bit, ta sẽ sinh ra **16 subkeys (mỗi key 48 bit)** để dùng cho 16 vòng Feistel.

### Quy trình:

1. Bỏ 8 bit parity → còn 56 bit (PC-1)
2. Chia thành 2 nửa:
   - C (28 bit)
   - D (28 bit)
3. Dịch trái (left shift) theo từng vòng
4. Ghép lại C + D
5. Áp dụng PC-2 → tạo subkey 48 bit


In [13]:
# Permuted Choice 1 (PC-1)
PC1 = [
57, 49, 41, 33, 25, 17, 9,
1, 58, 50, 42, 34, 26, 18,
10, 2, 59, 51, 43, 35, 27,
19, 11, 3, 60, 52, 44, 36,
63, 55, 47, 39, 31, 23, 15,
7, 62, 54, 46, 38, 30, 22,
14, 6, 61, 53, 45, 37, 29,
21, 13, 5, 28, 20, 12, 4
]

# Permuted Choice 2 (PC-2)
PC2 = [
14, 17, 11, 24, 1, 5,
3, 28, 15, 6, 21, 10,
23, 19, 12, 4, 26, 8,
16, 7, 27, 20, 13, 2,
41, 52, 31, 37, 47, 55,
30, 40, 51, 45, 33, 48,
44, 49, 39, 56, 34, 53,
46, 42, 50, 36, 29, 32
]

# Bảng dịch trái (Shift Schedule)
SHIFT_TABLE = [
1, 1, 2, 2,
2, 2, 2, 2,
1, 2, 2, 2,
2, 2, 2, 1
]

In [15]:
# Hàm hỗ trợ
def permute(block, table):
    return [block[i - 1] for i in table]

def left_shift(bits, n):
    return bits[n:] + bits[:n]

# Sinh 16 subkeys
def generate_keys(key_bits):
    # Bước 1: PC-1
    key = permute(key_bits, PC1)

    # Chia C và D
    C = key[:28]
    D = key[28:]

    keys = []

    for i in range(16):
        # Shift
        C = left_shift(C, SHIFT_TABLE[i])
        D = left_shift(D, SHIFT_TABLE[i])

        # Ghép lại
        combined = C + D

        # PC-2 → subkey 48 bit
        subkey = permute(combined, PC2)
        keys.append(subkey)

    return keys

- DES dùng 16 khóa con khác nhau
- Mỗi vòng Feistel dùng 1 subkey
- Khi giải mã: dùng key theo thứ tự ngược lại

## 4. Hàm f trong DES (Feistel Function)

Hàm f là thành phần chính trong mỗi vòng của DES.

### Input:
- R (32 bit)
- Subkey (48 bit)

### Output:
- 32 bit

### Các bước:

1. Expansion (E): 32 → 48 bit
2. XOR với subkey
3. Chia thành 8 block (6 bit)
4. Đi qua 8 S-box → mỗi block còn 4 bit
5. Ghép lại → 32 bit
6. Permutation (P)


### 4.1 S-box (Substitution Box)
DES có 8 S-box, mỗi S-box:

- Input: 6 bit
- Output: 4 bit

Cách hoạt động:
- 2 bit ngoài → chọn hàng
- 4 bit giữa → chọn cột

In [16]:
S_BOX = [
# S1
[
[14,4,13,1,2,15,11,8,3,10,6,12,5,9,0,7],
[0,15,7,4,14,2,13,1,10,6,12,11,9,5,3,8],
[4,1,14,8,13,6,2,11,15,12,9,7,3,10,5,0],
[15,12,8,2,4,9,1,7,5,11,3,14,10,0,6,13]
],

# S2
[
[15,1,8,14,6,11,3,4,9,7,2,13,12,0,5,10],
[3,13,4,7,15,2,8,14,12,0,1,10,6,9,11,5],
[0,14,7,11,10,4,13,1,5,8,12,6,9,3,2,15],
[13,8,10,1,3,15,4,2,11,6,7,12,0,5,14,9]
],

# S3
[
[10,0,9,14,6,3,15,5,1,13,12,7,11,4,2,8],
[13,7,0,9,3,4,6,10,2,8,5,14,12,11,15,1],
[13,6,4,9,8,15,3,0,11,1,2,12,5,10,14,7],
[1,10,13,0,6,9,8,7,4,15,14,3,11,5,2,12]
],

# S4
[
[7,13,14,3,0,6,9,10,1,2,8,5,11,12,4,15],
[13,8,11,5,6,15,0,3,4,7,2,12,1,10,14,9],
[10,6,9,0,12,11,7,13,15,1,3,14,5,2,8,4],
[3,15,0,6,10,1,13,8,9,4,5,11,12,7,2,14]
],

# S5
[
[2,12,4,1,7,10,11,6,8,5,3,15,13,0,14,9],
[14,11,2,12,4,7,13,1,5,0,15,10,3,9,8,6],
[4,2,1,11,10,13,7,8,15,9,12,5,6,3,0,14],
[11,8,12,7,1,14,2,13,6,15,0,9,10,4,5,3]
],

# S6
[
[12,1,10,15,9,2,6,8,0,13,3,4,14,7,5,11],
[10,15,4,2,7,12,9,5,6,1,13,14,0,11,3,8],
[9,14,15,5,2,8,12,3,7,0,4,10,1,13,11,6],
[4,3,2,12,9,5,15,10,11,14,1,7,6,0,8,13]
],

# S7
[
[4,11,2,14,15,0,8,13,3,12,9,7,5,10,6,1],
[13,0,11,7,4,9,1,10,14,3,5,12,2,15,8,6],
[1,4,11,13,12,3,7,14,10,15,6,8,0,5,9,2],
[6,11,13,8,1,4,10,7,9,5,0,15,14,2,3,12]
],

# S8
[
[13,2,8,4,6,15,11,1,10,9,3,14,5,0,12,7],
[1,15,13,8,10,3,7,4,12,5,6,11,0,14,9,2],
[7,11,4,1,9,12,14,2,0,6,10,13,15,3,5,8],
[2,1,14,7,4,10,8,13,15,12,9,0,3,5,6,11]
]
]

### 4.2 Hàm xử lý S-box

In [18]:
def sbox_substitution(bits48):
    result = []

    for i in range(8):
        block = bits48[i*6:(i+1)*6]

        row = (block[0] << 1) | block[5]
        col = (block[1] << 3) | (block[2] << 2) | (block[3] << 1) | block[4]

        val = S_BOX[i][row][col]

        # chuyển sang 4-bit
        bin_val = [int(x) for x in format(val, '04b')]
        result.extend(bin_val)

    return result

### 4.3 Hàm f hoàn chỉnh

In [19]:
def f_function(R, subkey):
    # 1. Expansion
    expanded = permute(R, E)

    # 2. XOR
    xor_result = [a ^ b for a, b in zip(expanded, subkey)]

    # 3. S-box
    sbox_result = sbox_substitution(xor_result)

    # 4. Permutation P
    final = permute(sbox_result, P)

    return final

- S-box là phần tạo phi tuyến (non-linearity)
- Đây là lý do DES không bị phá dễ bằng tuyến tính

## 5. Feistel Network và quá trình mã hóa DES

Sau khi có:
- Initial Permutation (IP)
- 16 subkeys
- Hàm f

Ta tiến hành 16 vòng Feistel.

### Quy trình mỗi vòng:

- L(i) = R(i-1)
- R(i) = L(i-1) XOR f(R(i-1), K(i))

### Sau 16 vòng:
- Đổi chỗ L và R
- Áp dụng Final Permutation (FP)


### 5.1 Hàm xử lý 1 block DES

In [20]:
def des_encrypt_block(block, keys):
    # Initial Permutation
    block = permute(block, IP)

    # Split
    L = block[:32]
    R = block[32:]

    # 16 rounds
    for i in range(16):
        new_L = R
        new_R = [l ^ f for l, f in zip(L, f_function(R, keys[i]))]

        L, R = new_L, new_R

    # Swap
    combined = R + L

    # Final Permutation
    cipher = permute(combined, FP)

    return cipher

### 5.2 Giải mã (Decrypt)

In [21]:
def des_decrypt_block(block, keys):
    # Đảo thứ tự keys
    reversed_keys = keys[::-1]
    return des_encrypt_block(block, reversed_keys)

### 5.3 Hàm hỗ trợ chuyển đổi

In [22]:
def string_to_bits(s):
    bits = []
    for c in s:
        binval = format(ord(c), '08b')
        bits.extend([int(b) for b in binval])
    return bits

def bits_to_string(bits):
    chars = []
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        chars.append(chr(int(''.join(map(str, byte)), 2)))
    return ''.join(chars)

### 5.4 Hàm encrypt/decrypt hoàn chỉnh

In [23]:
def des_encrypt(plaintext, key):
    plaintext_bits = string_to_bits(plaintext)
    key_bits = string_to_bits(key)

    keys = generate_keys(key_bits)

    cipher_bits = des_encrypt_block(plaintext_bits, keys)

    return cipher_bits


def des_decrypt(cipher_bits, key):
    key_bits = string_to_bits(key)

    keys = generate_keys(key_bits)

    plain_bits = des_decrypt_block(cipher_bits, keys)

    return bits_to_string(plain_bits)

### 5.5 Chạy thử chương trình

In [29]:
# In chương trình cho đẹp 1 tí :D
from IPython.display import display, HTML

def bits_to_hex(bits):
    return hex(int(''.join(map(str, bits)), 2))[2:].upper()

plaintext = "KHANHNGU"
key = "20224020"  

cipher_bits = des_encrypt(plaintext, key)
decrypted = des_decrypt(cipher_bits, key)

cipher_hex = bits_to_hex(cipher_bits)

html_content = f"""
<table style="border-collapse:collapse;border:1px solid black;">
    <tr>
        <th style="border:1px solid black;padding:8px;text-align:left;font-weight:bold;">Input</th>
        <th style="border:1px solid black;padding:8px;text-align:left;font-weight:bold;">Value</th>
    </tr>
    <tr>
        <td style="border:1px solid black;padding:8px;text-align:left;">Plaintext</td>
        <td style="border:1px solid black;padding:8px;text-align:left;">{plaintext}</td>
    </tr>
    <tr>
        <td style="border:1px solid black;padding:8px;text-align:left;">Key</td>
        <td style="border:1px solid black;padding:8px;text-align:left;">{key}</td>
    </tr>
    <tr>
        <td style="border:1px solid black;padding:8px;text-align:left;">Cipher (HEX)</td>
        <td style="border:1px solid black;padding:8px;text-align:left;">{cipher_hex}</td>
    </tr>
    <tr>
        <td style="border:1px solid black;padding:8px;text-align:left;">Decrypted</td>
        <td style="border:1px solid black;padding:8px;text-align:left;">{decrypted}</td>
    </tr>
</table>
"""

display(HTML(html_content))

Input,Value
Plaintext,KHANHNGU
Key,20224020
Cipher (HEX),693FF2306FE0ACE0
Decrypted,KHANHNGU
